# Install required packages

In [1]:
import math
!pip install gymnasium minigrid pettingzoo pymunk matplotlib numpy


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Display video inside Jupyter Notebook

In [2]:
import base64
import glob
import io
from IPython.display import HTML
from IPython import display


def show_video(folder, prefix, episode):
    mp4list = glob.glob(f'{folder}/{prefix}-episode-{episode}.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display.display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

#  [Frozen Lake Environment](https://gymnasium.farama.org/environments/toy_text/frozen_lake/)

In [3]:
import gymnasium as gym

env = gym.make("FrozenLake-v1", render_mode="rgb_array", is_slippery=False)  # replace with your environment

# Enable Video recording

More about available wrappers in Gymnasium: https://gymnasium.farama.org/api/wrappers/

List of all available wrappers
https://gymnasium.farama.org/api/wrappers/table/

In [4]:
from gymnasium.wrappers import RecordEpisodeStatistics, RecordVideo

prefix = "test"
folder = "video-folder"

trigger = lambda x: x % 10 == 0
#trigger = lambda x: x % 100 == 0 # Save a video after every 100 episodes

env = RecordVideo(env, video_folder=folder, name_prefix=prefix, episode_trigger=trigger)

C:\Users\Radu\Documents\College\AN4\intelligent-systems\.venv\Lib\site-packages\gymnasium\wrappers\rendering.py:293: UserWarning: WARN: Overwriting existing videos at C:\Users\Radu\Documents\College\AN4\intelligent-systems\q_learn\notebooks\video-folder folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


## Show observation spaces

Useful to see how observations and actions are stored

In [ ]:
print("Observation Space", env.observation_space)
print("Sample Observation", env.observation_space.sample())

print("Action Space", env.action_space)
print("Sample Action", env.action_space.sample())

# [Q Learning Algorithm](https://en.wikipedia.org/wiki/Q-learning)

## Training Loop

In [ ]:
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import math
import matplotlib.pyplot as plt

num_eval_episodes = 500

env = RecordEpisodeStatistics(env, buffer_length=num_eval_episodes)

avg_per_eps = 10

gamma = 0.99 # discount factor
learning_rate = 0.05


epsilon_start = 1.0
decay_rate=0.8

epsilon_decay_function = lambda ep: epsilon_start * math.exp((-2*ep * decay_rate) / num_eval_episodes)

number_of_actions = env.action_space.n # discrete space
number_of_states = env.observation_space.n # discrete space


q_table = np.zeros((number_of_states, number_of_actions))

rewards = []

history = []

for episode_num in range(num_eval_episodes):
    obs, info = env.reset()

    episode_over = False
    total_reward = 0
    while not episode_over:

        epsilon = max(epsilon_decay_function(episode_num), 0.05)

        if np.random.uniform() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[obs])

        next_obs, reward, terminated, truncated, info = env.step(action)

        # update q-table
        delta = (reward + gamma * np.max(q_table[next_obs]) - q_table[obs, action])

        q_table[obs, action] = q_table[obs, action] + learning_rate * delta

        obs = next_obs

        total_reward += reward
        episode_over = terminated or truncated

    rewards.append(total_reward)

    if episode_num % avg_per_eps == 0:
        history.append(np.mean(rewards))

env.close()

# Plot averages
plt.title("Average Reward")
plt.plot(avg_per_eps * np.arange(len(history) + 1), history)
plt.xlabel("Episode")
plt.ylabel("Average Reward")
plt.show()


np.save('q_table', q_table)

In [10]:
show_video(folder=folder, prefix=prefix, episode=10)

## Testing Phase

In [ ]:
import numpy as np

q_table = np.load('q_table.npy')

test_env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")

trigger = lambda x: True
#trigger = lambda x: x % 100 == 0 # Save a video after every 100 episodes

test_folder = "test-folder"

test_env = RecordVideo(env, video_folder=test_folder, name_prefix=prefix, episode_trigger=trigger)


num_test_episodes = 10

for episode_num in range(num_test_episodes):
    obs, info = test_env.reset()

    episode_over = False
    while not episode_over:

        action = np.argmax(q_table[obs]) # just greedy

        next_obs, reward, terminated, truncated, info = test_env.step(action)

        obs = next_obs

        episode_over = terminated or truncated
test_env.close()


In [ ]:
show_video(episode=9, folder=test_folder, prefix=prefix)